# Vision-language-action policy

Given a top-down 128×128 image and an instruction such as *"move to the red block"*, the policy picks the next discrete action: `left`, `right`, `up`, `down` or `grasp`.


Everything runs on CPU. A full run takes about 2 minutes on an Apple Silicon laptop after the one-time model download (~600 MB).


In [ ]:
import hashlib
import json
import math
import random
import time
from collections import Counter
from pathlib import Path
from typing import NamedTuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image as ShowImage, display
from PIL import Image, ImageDraw
from transformers import CLIPModel, CLIPProcessor

SEED = 0
TRAIN_STEPS, TEST_STEPS = 5000, 1000
GRID, IMAGE_SIZE = 4, 128            # 4x4 cells of 32 px; see section 9 for 8x8
CELL = IMAGE_SIZE // GRID
ACTIONS = ["left", "right", "up", "down", "grasp"]
COLORS = {"red": (220, 40, 40), "green": (40, 170, 60), "blue": (40, 80, 220)}
SLACK_STEPS = 4                      # success = grasp target within expert steps + 4
CLIP_ID = "openai/clip-vit-base-patch32"
CACHE, OUT = Path("cache"), Path("outputs")
CACHE.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
torch.manual_seed(SEED)
print("torch", torch.__version__)

## 1. Scenes and expert labels

There is no simulator beyond this cell. 

A scene is three numbers: gripper cell, the three block cells and the target colour. `move` updates the gripper cell and `render` draws it. The expert moves along the axis with the larger remaining distance (ties go to x) and grasps when on the target, so every state has exactly one label. `y` increases downwards, as in the image, so `up` is `y - 1`.

In [ ]:
class Scene(NamedTuple):
    gripper: tuple[int, int]
    blocks: tuple[tuple[int, int], ...]  # red, green, blue
    target: str

    @property
    def goal(self):
        return self.blocks[list(COLORS).index(self.target)]

    @property
    def distance(self):
        return abs(self.goal[0] - self.gripper[0]) + abs(self.goal[1] - self.gripper[1])

    @property
    def instruction(self):
        return f"move to the {self.target} block"


def random_scene(rng):
    cells = rng.sample([(x, y) for x in range(GRID) for y in range(GRID)], 4)
    return Scene(cells[0], tuple(cells[1:]), rng.choice(list(COLORS)))


def expert_action(s):
    dx, dy = s.goal[0] - s.gripper[0], s.goal[1] - s.gripper[1]
    if dx == dy == 0:
        return "grasp"
    if abs(dx) >= abs(dy):
        return "right" if dx > 0 else "left"
    return "down" if dy > 0 else "up"


def move(s, action):
    """Returns (next_scene, done, success). Grasp ends the episode."""
    if action == "grasp":
        return s, True, s.gripper == s.goal
    dx, dy = {"left": (-1, 0), "right": (1, 0), "up": (0, -1), "down": (0, 1)}[action]
    x = min(max(s.gripper[0] + dx, 0), GRID - 1)
    y = min(max(s.gripper[1] + dy, 0), GRID - 1)
    return s._replace(gripper=(x, y)), False, False


def render(s):
    img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (235, 235, 235))
    d = ImageDraw.Draw(img)
    for (x, y), rgb in zip(s.blocks, COLORS.values()):
        d.rectangle([x * CELL + 3, y * CELL + 3, x * CELL + CELL - 4, y * CELL + CELL - 4], fill=rgb)
    gx, gy = s.gripper  # gripper: thick black frame, drawn last so it shows over a block
    d.rectangle([gx * CELL, gy * CELL, gx * CELL + CELL - 1, gy * CELL + CELL - 1], outline=(0, 0, 0), width=CELL // 5)
    return img


# Sanity check: the expert solves every scene in exactly distance + 1 steps.
rng = random.Random(123)
for _ in range(2000):
    s, n, done = random_scene(rng), 0, False
    start = s
    while not done:
        s, done, ok = move(s, expert_action(s))
        n += 1
    assert ok and n == start.distance + 1
print("expert check passed on 2000 random scenes")

rng = random.Random(SEED)
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for ax in axes:
    s = random_scene(rng)
    ax.imshow(render(s))
    ax.set_title(f"{s.target} -> {expert_action(s)}", fontsize=9)
    ax.axis("off")
plt.show()

## 2. Demonstrations, split by block arrangement

Each episode starts from a random scene and follows the expert to the grasp.
Consecutive steps of one episode differ by one cell, so a random split by step
would put near-duplicates in train and test. The held-out set and every
evaluation scene use block arrangements that never appear in training.

In [ ]:
def expert_steps(scene):
    out, done = [], False
    while not done:
        a = expert_action(scene)
        out.append((scene, ACTIONS.index(a)))
        scene, done, _ = move(scene, a)
    return out


def demos(n_steps, seed, exclude=frozenset()):
    rng, out = random.Random(seed), []
    while len(out) < n_steps:
        s = random_scene(rng)
        if s.blocks not in exclude:
            out += expert_steps(s)
    return out[:n_steps]


def fresh_scenes(n, seed, exclude):
    rng, out = random.Random(seed), []
    while len(out) < n:
        s = random_scene(rng)
        if s.blocks not in exclude:
            out.append(s)
    return out


t0 = time.perf_counter()
train = demos(TRAIN_STEPS, SEED)
seen = frozenset(s.blocks for s, _ in train)
test = demos(TEST_STEPS, SEED + 1, exclude=seen)
seen |= {s.blocks for s, _ in test}
scenes_10 = fresh_scenes(10, SEED + 2, exclude=seen)
scenes_200 = fresh_scenes(200, SEED + 3, exclude=seen)
print(f"generated in {time.perf_counter() - t0:.2f}s")

assert not {s.blocks for s, _ in train} & {s.blocks for s, _ in test}
print(f"train: {len(train)} steps, {len({s.blocks for s, _ in train})} arrangements")
print(f"test:  {len(test)} steps, held-out arrangements")
print("train action counts:", {ACTIONS[a]: n for a, n in sorted(Counter(a for _, a in train).items())})

## 3. Frozen model embeddings (computed once, cached)

Image and text embeddings are model's projected 512-d vectors, L2-normalised.
model weights are frozen (`requires_grad_(False)`, `no_grad`); only the MLP
trains. Image embeddings are cached on disk, keyed by a hash of the exact
pixels, so re-running the notebook skips the ~20 s encoding step.

In [ ]:
clip = CLIPModel.from_pretrained(CLIP_ID).eval().requires_grad_(False)
processor = CLIPProcessor.from_pretrained(CLIP_ID)


@torch.no_grad()
def encode_images(images, batch=256):
    out = []
    for i in range(0, len(images), batch):
        px = processor(images=images[i : i + batch], return_tensors="pt")["pixel_values"]
        # vision_model + visual_projection == get_image_features, as a plain tensor on any transformers version
        out.append(clip.visual_projection(clip.vision_model(pixel_values=px).pooler_output))
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def encode_texts(texts):
    tok = processor(text=texts, return_tensors="pt", padding=True)
    pooled = clip.text_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).pooler_output
    return F.normalize(clip.text_projection(pooled), dim=-1)


def cached_image_embeddings(scenes):
    images = [render(s) for s in scenes]
    h = hashlib.sha256(CLIP_ID.encode())
    for im in images:
        h.update(im.tobytes())
    path = CACHE / f"{h.hexdigest()[:16]}.pt"
    if path.exists():
        return torch.load(path, weights_only=True)
    emb = encode_images(images)
    torch.save(emb, path)
    return emb


COLOR_NAMES = list(COLORS)
TEXT = encode_texts([f"move to the {c} block" for c in COLOR_NAMES])


def text_for(scenes):
    return TEXT[[COLOR_NAMES.index(s.target) for s in scenes]]


t0 = time.perf_counter()
x_img = cached_image_embeddings([s for s, _ in train])
t_img = cached_image_embeddings([s for s, _ in test])
x_txt, t_txt = text_for([s for s, _ in train]), text_for([s for s, _ in test])
y = torch.tensor([a for _, a in train])
t_y = torch.tensor([a for _, a in test])
print(f"embeddings ready in {time.perf_counter() - t0:.1f}s: images {tuple(x_img.shape)}, text {tuple(TEXT.shape)}")

## 4. Head and held-out next-action accuracy

`[image ; instruction] (1024) → 128 → ReLU → 5 logits`, Adam, a fixed 200 epochs.
No early stopping or model selection, so the held-out set is never used during
training.

Two accuracies are reported. **Exact** accuracy compares against the expert's
single label. **Valid-move** rate counts any action that reduces the distance
to the target (or grasps on it), since when both axes need to move either move
is equally good, and the expert's tie-break is arbitrary.

In [ ]:
def train_head(seed, epochs=200, lr=3e-3, batch=128):
    torch.manual_seed(seed)
    head = nn.Sequential(nn.Linear(2 * TEXT.shape[1], 128), nn.ReLU(), nn.Linear(128, len(ACTIONS)))
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    for _ in range(epochs):
        for b in torch.randperm(len(y)).split(batch):
            loss = F.cross_entropy(head(torch.cat([x_img[b], x_txt[b]], 1)), y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return head.eval()


@torch.no_grad()
def predict(head, img, txt):
    return head(torch.cat([img, txt], 1)).argmax(1)


def valid_move(s, a):
    if ACTIONS[a] == "grasp":
        return s.distance == 0
    return move(s, ACTIONS[a])[0].distance < s.distance


t0 = time.perf_counter()
head = train_head(SEED)
print(f"trained in {time.perf_counter() - t0:.1f}s, {sum(p.numel() for p in head.parameters()):,} parameters")

pred = predict(head, t_img, t_txt)
majority = Counter(y.tolist()).most_common(1)[0][0]
metrics = {
    "train_accuracy": (predict(head, x_img, x_txt) == y).float().mean().item(),
    "heldout_accuracy": (pred == t_y).float().mean().item(),
    "heldout_valid_move_rate": sum(valid_move(s, a) for (s, _), a in zip(test, pred.tolist())) / len(test),
    "heldout_majority_baseline": (t_y == majority).float().mean().item(),
}
for k, v in metrics.items():
    print(f"{k:<28}{v:.3f}")

print("\nconfusion (rows = expert, cols = predicted)")
print(" " * 8 + "".join(f"{a:>7}" for a in ACTIONS))
for i, a in enumerate(ACTIONS):
    row = [((t_y == i) & (pred == j)).sum().item() for j in range(len(ACTIONS))]
    print(f"{a:>7} " + "".join(f"{n:>7}" for n in row))

## 5. Closed-loop success on fresh scenes

The policy sees only the rendered frame and the instruction at each step. It
succeeds if it grasps on the named block within the expert's step count plus
4. A grasp anywhere else ends the episode as a failure.

Scenes run in lockstep so each step is one batched CLIP call. Besides the 10
scenes the task asks for, 200 scenes give a usable confidence interval. Three
comparison policies show what the numbers mean:

- **sampled**: the same head, sampling the action from its softmax instead of
  taking the argmax. This tests whether failures are loops that randomness can break.
- **random**: uniform over the 5 actions.
- **no instruction**: the same trained head with the instruction embedding
  replaced by the mean of all three. It sees the image but not which block to go to.
- **expert**: the labelling rule. Should be 1.0, as a check that the harness is correct.

In [ ]:
def rollout(policy, scenes):
    eps = [{"frames": [s], "actions": [], "done": False, "success": False} for s in scenes]
    while active := [e for e in eps if not e["done"] and len(e["actions"]) < e["frames"][0].distance + 1 + SLACK_STEPS]:
        for e, a in zip(active, policy([e["frames"][-1] for e in active])):
            s, e["done"], e["success"] = move(e["frames"][-1], a)
            e["actions"].append(a)
            if not e["done"]:
                e["frames"].append(s)
    return eps


def outcome(e):
    if e["success"]:
        return "success"
    if not e["done"]:
        # The policy is deterministic and memoryless, so a repeated frame means it is in a loop.
        return "timeout: pushing into wall" if e["frames"][-1] == e["frames"][-2] else "timeout: cycling"
    last = e["frames"][-1]
    on = [c for c, cell in zip(COLORS, last.blocks) if cell == last.gripper]
    return f"grasped {on[0]} block" if on else "grasped empty cell"


def clip_policy(head, text_fn=text_for):
    return lambda scenes: [ACTIONS[i] for i in predict(head, encode_images([render(s) for s in scenes]), text_fn(scenes))]


def sampled_policy(head, seed):
    g = torch.Generator().manual_seed(seed)

    def pol(scenes):
        with torch.no_grad():
            logits = head(torch.cat([encode_images([render(s) for s in scenes]), text_for(scenes)], 1))
        return [ACTIONS[i] for i in torch.multinomial(logits.softmax(1), 1, generator=g).squeeze(1).tolist()]

    return pol


def mean_text(scenes):
    return TEXT.mean(0, keepdim=True).expand(len(scenes), -1)


_rng = random.Random(SEED)
random_policy = lambda scenes: [_rng.choice(ACTIONS) for _ in scenes]
expert_policy = lambda scenes: [expert_action(s) for s in scenes]


def success_rate(eps):
    return sum(e["success"] for e in eps) / len(eps)


def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0, c - h), min(1, c + h)


t0 = time.perf_counter()
policy = clip_policy(head)
eps_10 = rollout(policy, scenes_10)
eps_200 = rollout(policy, scenes_200)
metrics["success_rate_10"] = success_rate(eps_10)
metrics["success_rate_200"] = success_rate(eps_200)
metrics["success_rate_200_ci95"] = wilson(sum(e["success"] for e in eps_200), 200)
metrics["success_rate_200_sampled"] = success_rate(rollout(sampled_policy(head, SEED), scenes_200))
metrics["success_rate_200_no_instruction"] = success_rate(rollout(clip_policy(head, mean_text), scenes_200))
metrics["success_rate_200_random"] = success_rate(rollout(random_policy, scenes_200))
metrics["success_rate_200_expert"] = success_rate(rollout(expert_policy, scenes_200))
print(f"rollouts in {time.perf_counter() - t0:.1f}s\n")

print(f"10 fresh scenes : {metrics['success_rate_10']:.0%}  " + " ".join("✓" if e["success"] else "✗" for e in eps_10))
lo, hi = metrics["success_rate_200_ci95"]
print(f"200 fresh scenes: {metrics['success_rate_200']:.1%}  (95% CI {lo:.1%}–{hi:.1%})")
for name in ["sampled", "no_instruction", "random", "expert"]:
    print(f"  {name:<15}{metrics['success_rate_200_' + name]:.1%}")
print("failure outcomes (200):", dict(Counter(outcome(e) for e in eps_200 if not e["success"])))

## 6. Five-step episode on an unseen arrangement

Fresh scenes where the expert needs exactly five steps (four moves, then grasp).
The success rate over the first 20 such scenes is printed, and the first
successful one is shown and saved to `outputs/demo.gif`. Showing the first
success is a choice made for the demo; the rate printed next to it is the
unselected number.

In [ ]:
def save_gif(e, path, scale=2):
    captions = e["actions"] + ([] if e["done"] else ["(step budget used up)"])
    frames = []
    for i, s in enumerate(e["frames"]):
        cap = captions[i] if i < len(captions) else ""
        if i == len(e["frames"]) - 1 and e["done"]:
            cap += f"  ->  {outcome(e)}"
        board = render(s).resize((IMAGE_SIZE * scale,) * 2, Image.NEAREST)
        frame = Image.new("RGB", (board.width, board.height + 36), "white")
        frame.paste(board, (0, 36))
        d = ImageDraw.Draw(frame)
        d.text((4, 2), s.instruction, fill="black")
        d.text((4, 18), f"step {i + 1}: {cap}", fill=(90, 90, 90))
        frames.append(frame)
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=800, loop=0)
    return path


five_step = [s for s in fresh_scenes(2000, SEED + 4, exclude=seen) if s.distance == 4][:20]
five_eps = rollout(policy, five_step)
metrics["success_rate_five_step_20"] = success_rate(five_eps)
print(f"five-step scenes solved: {sum(e['success'] for e in five_eps)}/20")
demo = next((e for e in five_eps if e["success"]), five_eps[0])
demo_scene = demo["frames"][0]
print(demo_scene.instruction)
print("policy:", demo["actions"], "->", outcome(demo))
print("expert:", [ACTIONS[a] for _, a in expert_steps(demo_scene)])
display(ShowImage(filename=save_gif(demo, OUT / "demo.gif")))

## 7. Failure case

The first failed episode among the 10 fresh scenes, or among the 200 if all 10
succeeded. Saved to `outputs/failure.gif`.

In [ ]:
failure = next((e for e in eps_10 + eps_200 if not e["success"]), None)
if failure is None:
    print("no failures in 210 episodes")
else:
    s0 = failure["frames"][0]
    print(f"{s0.instruction}: gripper {s0.gripper}, target {s0.goal}, blocks {dict(zip(COLORS, s0.blocks))}")
    print("policy:", failure["actions"], "->", outcome(failure))
    print("expert:", [ACTIONS[a] for _, a in expert_steps(s0)])
    display(ShowImage(filename=save_gif(failure, OUT / "failure.gif")))